# 4.2 — Word Embeddings avec spaCy

## Objectifs pédagogiques
À la fin de ce notebook, vous serez capable de :
- charger un modèle linguistique pré-entraîné contenant des vecteurs de mots ;
- accéder aux embeddings d'un mot, d'un token et d'un document ;
- mesurer la similarité sémantique entre deux mots ou deux textes ;
- vérifier si un mot possède un vecteur exploitable ;
- illustrer une analogie vectorielle avec l'exemple **king − man + woman ≈ queen**.

> **Idée essentielle :** un embedding représente un mot par un vecteur numérique dense. Dans un espace vectoriel appris à partir de données textuelles, des mots utilisés dans des contextes proches peuvent avoir des représentations proches.

## 1. Installation et chargement du modèle

Le modèle `en_core_web_lg` est un grand modèle anglais de spaCy qui contient des vecteurs statiques de 300 dimensions.

Dans un terminal, exécutez les commandes suivantes (une seule fois) :

```bash
python -m pip install -U spacy scipy matplotlib
python -m spacy download en_core_web_lg
```

Si vous travaillez dans Jupyter Notebook, vous pouvez utiliser `!` devant les commandes.

In [ ]:
# Installation depuis un notebook Jupyter (à exécuter si nécessaire)
# %pip install -U spacy scipy matplotlib
# !python -m spacy download en_core_web_lg

In [ ]:
import spacy

# Chargement du modèle anglais pré-entraîné avec des vecteurs statiques.
nlp = spacy.load("en_core_web_lg")

print("Modèle chargé :", nlp.meta["name"])
print("Version spaCy :", spacy.__version__)
print("Dimensions des vecteurs :", nlp.vocab.vectors_length)

## 2. Accéder aux vecteurs des mots

Un token possède un attribut `.vector` qui retourne son embedding sous forme de tableau NumPy.

Les coordonnées ne sont pas directement interprétables une par une : c'est la position relative des vecteurs dans l'espace appris qui porte l'information.

In [ ]:
# Récupération des vecteurs de deux mots
brave_vector = nlp("brave")[0].vector
lion_vector = nlp("lion")[0].vector

print("Vecteur de 'brave' :", brave_vector)
print("Vecteur de 'lion'  :", lion_vector)
print("Nombre de dimensions :", len(lion_vector))

## 3. Vecteur d'un document

spaCy calcule également un vecteur pour un document (`Doc`). Pour les modèles à vecteurs statiques, ce vecteur de document est essentiellement une représentation agrégée des vecteurs de ses tokens.

Il donne une représentation globale du texte, mais ne conserve pas à lui seul l'ordre des mots.

In [ ]:
doc = nlp("The quick brown fox jumped over the lazy dogs.")

print("Texte :", doc.text)
print("Dimensions du vecteur du document :", doc.vector.shape)
print("Vecteur du document :\n", doc.vector)

## 4. Similarité entre les mots

La méthode `.similarity()` calcule une mesure de similarité cosinus entre deux représentations vectorielles.

- Une valeur proche de **1** indique des vecteurs orientés dans des directions proches.
- Une valeur proche de **0** indique peu d'alignement.
- Une valeur négative indique des directions opposées.

**Attention :** la similarité cosinus mesure une proximité géométrique, pas nécessairement une synonymie. Deux mots associés par le contexte peuvent être similaires sans avoir le même sens.

In [ ]:
# Comparaison de chaque paire de mots de la liste
tokens = nlp("lion cat pet")

for token1 in tokens:
    for token2 in tokens:
        print(f"{token1.text:>6s} / {token2.text:<6s} : "
              f"{token1.similarity(token2):.3f}")

### 4.1 Comparer des paires de mots

Les exemples suivants permettent de constater que la similarité dépend des relations apprises dans le corpus. Les résultats sont propres au modèle utilisé.

In [ ]:
# Paires de mots à comparer
pairs = [
    ("man", "woman"),
    ("man", "tree"),
    ("woman", "moon"),
    ("woman", "flower"),
    ("lion", "tiger"),
    ("lion", "dandelion"),
    ("some", "same"),
    ("piece", "peace"),
    ("love", "hate"),
    ("love", "flower"),
]

for word1, word2 in pairs:
    score = nlp(word1).similarity(nlp(word2))
    print(f"{word1:>10s} / {word2:<10s} : {score:.3f}")

### 4.2 Similarité entre des phrases

La même méthode peut comparer les vecteurs de documents. Les phrases peuvent être proches dans l'espace vectoriel même si certains mots diffèrent.

La mesure ne remplace toutefois pas une analyse fine du sens, de la négation ou de la composition syntaxique.

In [ ]:
sentences = [
    "I love school",
    "I hate school",
    "This film is awesome. I love it.",
    "This film is boring. I hate it."
]

for i in range(0, len(sentences), 2):
    score = nlp(sentences[i]).similarity(nlp(sentences[i + 1]))
    print(f"Phrase 1 : {sentences[i]}")
    print(f"Phrase 2 : {sentences[i + 1]}")
    print(f"Similarité cosinus : {score:.3f}\n")

## 5. Vocabulaire et couverture des vecteurs

Le vocabulaire du modèle contient des entrées lexicales et une table de vecteurs. Tous les mots rencontrés dans un texte ne possèdent pas nécessairement un vecteur pré-entraîné.

- `has_vector` : indique si un vecteur est disponible ;
- `vector_norm` : norme du vecteur ;
- `is_oov` : indique si le mot est hors vocabulaire (out of vocabulary) pour le modèle.

In [ ]:
# Taille du vocabulaire et dimensions de la table des vecteurs
print("Nombre d'entrées dans le vocabulaire :", len(nlp.vocab))
print("Nombre de vecteurs :", len(nlp.vocab.vectors))
print("Forme de la table des vecteurs :", nlp.vocab.vectors.shape)

In [ ]:
# Vérifier la disponibilité des vecteurs pour différents mots
tokens = nlp("dog cat nargle hesham lksdvsk")

for token in tokens:
    print(
        f"{token.text:10s} | "
        f"has_vector={token.has_vector!s:5s} | "
        f"vector_norm={token.vector_norm:8.3f} | "
        f"is_oov={token.is_oov}"
    )

## 6. Analogie vectorielle : king − man + woman ≈ queen

Une illustration classique des embeddings est l'analogie :

\[
\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}
\]

L'intuition est la suivante : on retire au vecteur *king* une composante associée à *man*, puis on ajoute une composante associée à *woman*. On recherche ensuite les mots dont le vecteur est le plus proche du vecteur obtenu.

Cette relation est une régularité observée dans certains espaces d'embeddings ; elle n'est ni une règle linguistique universelle ni une garantie pour tous les modèles.

In [ ]:
import numpy as np
from scipy.spatial.distance import cosine

# Récupérer les vecteurs des trois mots de l'analogie.
king = nlp.vocab["king"].vector
man = nlp.vocab["man"].vector
woman = nlp.vocab["woman"].vector

# Calcul de l'analogie : king - man + woman
target_vector = king - man + woman

# Quelques candidats pour illustrer le classement.
candidate_words = [
    "cat", "apple", "queen", "castle", "sea", "shell", "orange",
    "phone", "angry", "book", "white", "land", "study", "crown",
    "prince", "dog", "princess", "elizabeth"
]

# Calculer la similarité cosinus entre le vecteur cible et chaque candidat.
scores = []
for word in candidate_words:
    candidate_vector = nlp.vocab[word].vector

    # Éviter un calcul non défini si un vecteur est nul.
    if np.linalg.norm(candidate_vector) == 0 or np.linalg.norm(target_vector) == 0:
        continue

    similarity = 1 - cosine(target_vector, candidate_vector)
    scores.append((word, similarity))

# Trier du candidat le plus proche au moins proche.
scores.sort(key=lambda item: item[1], reverse=True)

print("Analogie : king - man + woman")
print("\nCandidats les plus proches :")
for word, score in scores[:10]:
    print(f"{word:12s} : {score:.4f}")

### 6.1 Recherche dans tout le vocabulaire du modèle (amélioration)

La liste de candidats précédente est volontairement courte : elle sert à comprendre le calcul, mais elle peut empêcher de trouver le meilleur mot. Dans la cellule suivante, on utilise la table des vecteurs du modèle pour rechercher les voisins les plus proches parmi les vecteurs disponibles.

On exclut les mots de l'analogie et on utilise la similarité cosinus. Cette recherche est plus représentative, mais reste dépendante du vocabulaire et des données d'entraînement du modèle.

In [ ]:
# Recherche des voisins les plus proches dans la table de vecteurs de spaCy.
# Cette cellule peut prendre quelques secondes selon la machine.

vectors = nlp.vocab.vectors.data
keys = nlp.vocab.vectors.keys()

# Vérifier que le modèle contient bien des vecteurs.
if vectors.size == 0:
    raise ValueError("Aucun vecteur n'est disponible dans le modèle chargé.")

# Normalisation L2 : le produit scalaire devient la similarité cosinus.
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
normalized_vectors = np.divide(
    vectors,
    norms,
    out=np.zeros_like(vectors, dtype=np.float32),
    where=norms != 0
)

target_norm = np.linalg.norm(target_vector)
if target_norm == 0:
    raise ValueError("Le vecteur cible est nul : similarité cosinus impossible.")

normalized_target = target_vector / target_norm
all_scores = normalized_vectors @ normalized_target

# Exclure les mots utilisés dans l'opération.
excluded = {"king", "man", "woman"}
ranking = np.argsort(all_scores)[::-1]

results = []
for index in ranking:
    lexeme = nlp.vocab[keys[index]]
    word = lexeme.text

    if word.lower() in excluded or not word.isalpha():
        continue

    results.append((word, float(all_scores[index])))
    if len(results) == 10:
        break

for word, score in results:
    print(f"{word:15s} : {score:.4f}")

## 7. Visualisation de l'analogie dans un plan 2D

Les embeddings de `en_core_web_lg` ont 300 dimensions. Pour les représenter dans un plan, on peut projeter les quatre vecteurs sur deux dimensions avec PCA.

**Important :** la projection 2D perd une partie de l'information contenue dans les 300 dimensions. La relation vectorielle doit donc être calculée dans l'espace original, puis les vecteurs sont projetés uniquement pour l'affichage.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Calcul de la relation dans l'espace original (300 dimensions).
queen_target = king - man + woman

# Réunir les vecteurs à projeter.
words_for_plot = ["man", "king", "woman", "queen", "résultat"]
vectors_for_plot = np.vstack([
    man,
    king,
    woman,
    nlp.vocab["queen"].vector,
    queen_target
])

# PCA réduit les 300 dimensions à 2 composantes pour la visualisation.
pca = PCA(n_components=2, random_state=42)
points_2d = pca.fit_transform(vectors_for_plot)

# Tracer les points et leurs étiquettes.
plt.figure(figsize=(9, 7))
for label, (x, y) in zip(words_for_plot, points_2d):
    plt.scatter(x, y, s=90)
    plt.annotate(label, (x, y), xytext=(7, 7),
                 textcoords="offset points", fontsize=11)

plt.title("Projection PCA des vecteurs de l'analogie")
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Variance expliquée par les deux composantes :",
      f"{pca.explained_variance_ratio_.sum():.1%}")

## 8. Remarque sur les exemples en arabe

Le modèle `en_core_web_lg` est un modèle anglais. Il ne faut donc pas interpréter ses résultats sur des mots ou des phrases arabes comme des embeddings arabes fiables.

Les exemples arabes ci-dessous sont conservés comme illustration de la différence entre traitement du texte et disponibilité de vecteurs. Pour obtenir des représentations arabes pertinentes, il faut choisir un modèle entraîné pour l'arabe ou un modèle multilingue adapté.

In [ ]:
# Exemple de texte arabe : le pipeline anglais peut tokeniser le texte,
# mais ses vecteurs ne constituent pas des représentations arabes fiables.
arabic_text = "ذهب محمد إلى الجامعة"
arabic_doc = nlp(arabic_text)

print("Texte :", arabic_doc.text)
for token in arabic_doc:
    print(token.text, "| has_vector =", token.has_vector,
          "| is_oov =", token.is_oov)

## 9. À retenir

1. Un **word embedding** associe à un mot un vecteur dense.
2. Les vecteurs de spaCy peuvent être consultés avec `.vector`.
3. La similarité cosinus compare l'orientation de deux vecteurs.
4. Un vecteur de document fournit une représentation globale du texte, avec les limites d'une agrégation.
5. Une analogie vectorielle se calcule par addition et soustraction de vecteurs.
6. Une projection 2D (PCA) sert à visualiser des données de grande dimension, mais elle ne préserve pas toutes les distances.

### Questions de réflexion
- Pourquoi deux mots associés dans les mêmes contextes peuvent-ils avoir une similarité élevée sans être synonymes ?
- Quelle est la différence entre un mot hors vocabulaire et un mot qui possède un vecteur nul ?
- Pourquoi faut-il calculer l'analogie dans les 300 dimensions avant de la projeter en 2D ?
- Quels biais peuvent être présents dans les analogies apprises par un modèle ?